# XRP/USDC — Colab GPU research
Open this notebook in VS Code. Select Kernel → Colab, sign in, and select a GPU server.
Right-click `data/processed/colab/crypto_agent_colab.zip` in VS Code and select
**Upload to Colab**. This notebook uses temporary runtime storage; download the results ZIP before disconnecting.
Run setup cells in order. The training cell defaults to disabled, even with Run All.
Only public candles, research source, and split metadata are in the bundle.
These are development experiments, not a fresh final test or an approved trading model.


In [2]:
print('Using Colab runtime storage. Export results before disconnecting.')


Using Colab runtime storage. Export results before disconnecting.


In [3]:
from pathlib import Path
import hashlib, json, os, sys, zipfile, tempfile, subprocess, shutil

archive = Path('/content/crypto_agent_transfer/crypto_agent_colab.zip')
uploaded = Path('/content/crypto_agent_colab.zip')
if uploaded.is_file():
    archive.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(uploaded, archive)
if not archive.is_file():
    raise FileNotFoundError('Use Upload to Colab for the ZIP or place it in My Drive/Crypto_agent.')
work = Path(tempfile.mkdtemp(prefix='crypto_agent_', dir='/content'))
with zipfile.ZipFile(archive) as bundle:
    manifest = json.loads(bundle.read('manifest.json'))
    if set(bundle.namelist()) != set(manifest) | {'manifest.json'}:
        raise ValueError('Unexpected bundle contents.')
    for name, expected in manifest.items():
        target = (work / name).resolve()
        if not target.is_relative_to(work.resolve()):
            raise ValueError('Unsafe archive path.')
        content = bundle.read(name)
        if hashlib.sha256(content).hexdigest() != expected:
            raise ValueError(f'Checksum mismatch: {name}')
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(content)
os.chdir(work)
sys.path.insert(0, str(work))
print('Verified and extracted research bundle:', work)


Verified and extracted research bundle: /content/crypto_agent_5qcs5v_6


In [4]:
# Keep Colab's supplied TensorFlow/CUDA stack; install only research helpers.
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'pandas>=2.2,<4', 'scikit-learn>=1.5,<2',
                'xgboost>=2,<4', 'joblib>=1.3,<2'], check=True)
# Set device mode before importing the research module or initializing TensorFlow.
os.environ['CRYPTO_TF_DEVICE'] = 'gpu'
os.environ.pop('CUDA_VISIBLE_DEVICES', None)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if not gpus:
    raise RuntimeError('No GPU detected. Select a GPU Colab server and restart this notebook.')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
print('TensorFlow:', tf.__version__, 'GPUs:', gpus)


TensorFlow: 2.21.0 GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [5]:
from src import tensorflow_research as research
reference = Path('models/coinbase-20261020T121949544468Z/metadata.json')
output_root = Path('/content/research_models')
output_root.mkdir(parents=True, exist_ok=True)
probe = output_root / '.write_check'
probe.write_text('ok')
probe.unlink()
os.environ['CRYPTO_RESEARCH_OUTPUT'] = str(output_root)
plan = research.prepare(reference, lookback=48, epochs=25)
(output_root / 'preparation_report.json').write_text(json.dumps(plan, indent=2))
print('Ready for review. No market-data training has started.')


{
  "status": "prepared_not_trained",
  "lookback_hours": 48,
  "features": [
    "return_1h",
    "return_3h",
    "return_6h",
    "return_24h",
    "sma_6",
    "sma_24",
    "sma_72",
    "ema_12",
    "ema_26",
    "rsi_14",
    "macd",
    "macd_signal",
    "volatility_24h",
    "volume_change",
    "relative_volume",
    "candle_range",
    "candle_body",
    "hour_sin",
    "hour_cos",
    "weekday_sin",
    "weekday_cos"
  ],
  "eligible_sequences": 19864,
  "provenance": {
    "reference_metadata": "models/coinbase-20261020T121949544468Z/metadata.json",
    "reference_sha256": "742e6af3446d0c79eea3ae5770f6b4a2ef0cf1125b07382341295855389c43dd",
    "last_allowed_raw_candle": "2025-10-23T10:00:00+00:00",
    "excluded_original_validation_start": "2025-10-23 11:00:00+00:00",
    "excluded_original_test_start": "2026-04-22 11:00:00+00:00",
    "source_sha256": "755391d16506214447e4942345e8f34201be1b8b3c3a7365f190e1c4d9d1fc9d"
  },
  "quality": {
    "input_rows": 19983,
    "inv

## Start training only after reviewing the preparation report
Change `START_TRAINING` to `True` only when ready, then run the next cell.
Models, best-epoch checkpoints and logs save in runtime storage; download the final results ZIP before disconnecting.
If a runtime disconnects, saved checkpoints remain, but automatic optimizer/epoch resume
is not implemented. A new training call creates a separate run; it does not overwrite the old one.
Do not repeatedly adjust settings based on development evaluation results.


In [6]:
START_TRAINING = False
if START_TRAINING:
    run_directory = research.train(reference, lookback=48, epochs=25)
    print('Saved results:', run_directory)
else:
    print('Training is disabled. Review the plan before setting START_TRAINING=True.')


Training directory: /content/research_models/tensorflow-20261022T052236408074Z
Development sequences: 19864
Cutoff: 2025-10-23T10:00:00+00:00
fold_1/gru epoch 1: loss=0.446845, validation_loss=0.727679
fold_1/gru epoch 2: loss=0.444533, validation_loss=0.725879
fold_1/gru epoch 3: loss=0.443323, validation_loss=0.724950
fold_1/gru epoch 4: loss=0.442507, validation_loss=0.724296
fold_1/gru epoch 5: loss=0.442134, validation_loss=0.723942
fold_1/gru epoch 6: loss=0.441208, validation_loss=0.723510
fold_1/gru epoch 7: loss=0.441155, validation_loss=0.723500
fold_1/gru epoch 8: loss=0.440675, validation_loss=0.723384
fold_1/gru epoch 9: loss=0.440807, validation_loss=0.722935
fold_1/gru epoch 10: loss=0.440381, validation_loss=0.723072
fold_1/gru epoch 11: loss=0.439597, validation_loss=0.722831
fold_1/gru epoch 12: loss=0.439710, validation_loss=0.722453
fold_1/gru epoch 13: loss=0.439385, validation_loss=0.722069
fold_1/gru epoch 14: loss=0.439119, validation_loss=0.722145
fold_1/gru ep

In [8]:
# Display results only if this notebook completed a training run.
if 'run_directory' in globals():
    metadata = json.loads((run_directory / 'metadata.json').read_text())
    import pandas as pd
    display(pd.DataFrame(metadata['pooled_development']).T)
    print('Research only; no model promotion or trading.')
    import shutil
    export_zip = shutil.make_archive('/content/' + run_directory.name, 'zip', root_dir=run_directory.parent, base_dir=run_directory.name)
    print('Download this archive from Colab Contents:', export_zip)
else:
    print('No completed training run in this session.')


,samples,mae,rmse,bias,directional_accuracy,directional_samples,forecast_above_2pct_count,forecast_above_2pct_coverage,selective_directional_accuracy,error_quantiles
zero_return,5960,0.006185,0.009862,-0.000071,0.0,5879,0,0.0,None,"{'p05': -0.013636624447690348, 'p50': -4.11573..."
persistence,5960,0.009054,0.014034,-0.000003,0.489369,5879,279,0.046812,0.397112,"{'p05': -0.020302204037696014, 'p50': 0.000230..."
gru,5960,0.00615,0.009893,0.000327,0.52594,5879,0,0.0,None,"{'p05': -0.012998276283176203, 'p50': 0.000147..."
lstm,5960,0.006162,0.009894,0.000316,0.52628,5879,0,0.0,None,"{'p05': -0.012833450237197876, 'p50': 0.000133..."
xgboost,5960,0.006144,0.009835,-0.000007,0.516925,5879,0,0.0,None,"{'p05': -0.013421452758837948, 'p50': -5.00882..."
selected,5960,0.006158,0.009887,0.000325,0.52713,5879,0,0.0,None,"{'p05': -0.012782292518197303, 'p50': 0.000144..."


Research only; no model promotion or trading.
Download this archive from Colab Contents: /content/tensorflow-20261022T052236408074Z.zip
